# Subjectesis: Qwen3.5-4B on Kaggle T4 x2

**Fresh Kaggle session. Accelerator: GPU T4 x2. Internet: ON. Then Run All.**

This version uses the official post-trained **Qwen/Qwen3.5-4B**, not the Base variant,
with a pinned checkpoint and **4-bit NF4 QLoRA**. It keeps your exact revision-4 dataset
and all **30,266 Subjectesis examples**. Default: one Subjectesis adapter, one epoch.
The answer-only comparison remains optional.

**Both T4s train the same adapter:** each GPU holds a quantized copy of the model,
processes different examples, and synchronizes gradients. Their VRAM is not pooled.

## Included fixes

- Compatible **TorchAO 0.16.0** is installed automatically; no separate repair cell.
- The 4B model's **shared input/output embeddings** are preserved and checked.
- Model download is approximately **9.32 GB**, with storage calculated from the 4B files.
- Pip installation does not retain wheel caches. No dataset or old model cache is deleted.
- Existing CUDA-enabled PyTorch is retained, with no JAX/CUDA patching.

The checkpoint downloads in its original form before being quantized on the GPU.
Allow approximately **13 GiB free disk after setup** for the model and checkpoint/output reserve.
A fresh session avoids carrying the earlier 9B/Gemma caches.

**There is no five-hour guarantee.** The notebook reports actual step time and a running
estimate. Checkpoints permit resuming. `paused_resumable` does not mean the epoch completed.

Do not paste earlier repair cells into this notebook. No additional dataset upload or paid API.


## 1. Settings and embedded source

Keep the defaults to train one Subjectesis adapter. `TRAIN_BASELINE=True` adds a second,
separately trained answer-only adapter. `SMOKE_ONLY=True` runs the real model integration
test without the full epoch. The default four-hour cap saves and pauses unfinished training.
An optional Kaggle secret named `HF_TOKEN` can authenticate public model downloads.


In [1]:
from pathlib import Path
import zipfile

SRC = Path("/kaggle/input/datasets/santhoshkumares/resume-arhcive")
OUT = Path("/kaggle/working/resume_checkpoint.zip")

assert (SRC / "recovery_manifest.json").exists()
assert (SRC / "settings.json").exists()
assert (SRC / "runs").exists()

with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in SRC.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(SRC).as_posix())

print("Created:", OUT)
print("Size MB:", round(OUT.stat().st_size / 1024**2, 1))

Created: /kaggle/working/resume_checkpoint.zip
Size MB: 399.8


In [2]:
RUN_FOLDER = "subjectesis_qwen35_4b_t4x2_v1"
TRAIN_BASELINE = False       # True also trains the separate matched answer-only adapter.
SMOKE_ONLY = False           # False: real-model checks, then training.
EPOCHS = 1
MAX_SESSION_HOURS = 4        # Graceful checkpoint/pause. 0 removes this voluntary cap.
RESUME_ARCHIVE = "/kaggle/working/resume_checkpoint.zip"        # Next session: "/kaggle/input/.../subjectesis_qwen35_4b_outputs.zip"

from pathlib import Path
import base64, hashlib, io, os, sys, zipfile

if not RUN_FOLDER or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in RUN_FOLDER):
    raise ValueError("Use lowercase letters, numbers, hyphens or underscores in RUN_FOLDER.")
ROOT = Path(os.environ.get("SUBJECTESIS_NOTEBOOK_BASE", "/kaggle/working")) / RUN_FOLDER
ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD_B64 = '...'
PAYLOAD_SHA256 = '2a56f29d7ec7c55249d0d75afc4914551b00d3762d40cd3971243ca2efe5f996'

payload = base64.b64decode(PAYLOAD_B64, validate=True)
if hashlib.sha256(payload).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError("Notebook payload checksum failed")
# Embedded archive is authored with relative regular files only. Protect existing edits.
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise ValueError("Unsafe payload path")
        dest = ROOT / rel
        if not dest.resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Payload path escaped run directory")
        if dest.exists() and dest.read_bytes() != z.read(name):
            raise RuntimeError(f"Existing source differs: {name}. Use a new RUN_FOLDER; do not overwrite working checkpoints.")
    for name in z.namelist():
        dest = ROOT / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            dest.write_bytes(z.read(name))
sys.path.insert(0, str(ROOT / "code"))
from common import write_json, restore_runs, read_json, package_outputs
from launch import stream, child_env

settings = {
    "model_id": "Qwen/Qwen3.5-4B",
    "model_revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    "training_method": "NF4_QLoRA",
    "train_baseline": bool(TRAIN_BASELINE),
    "epochs": int(EPOCHS), "rank": 8, "alpha": 16,
    "learning_rate": 1e-4, "seed": 42,
    "effective_batch_size": 8, "max_length": 2048,
    "warmup_fraction": 0.03, "loss_chunk_size": 32,
    "save_steps": 25, "eval_steps": 500,
    "max_session_hours": float(MAX_SESSION_HOURS)
}
if (ROOT / "settings.json").exists() and read_json(ROOT / "settings.json") != settings:
    raise RuntimeError("Existing run settings differ. Use a new RUN_FOLDER, rather than overwrite a run.")
write_json(ROOT / "settings.json", settings)
if RESUME_ARCHIVE:
    restore_runs(Path(RESUME_ARCHIVE), ROOT)
    print("Verified checkpoints restored from the recovery archive.")
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret("HF_TOKEN")
    if secret:
        os.environ["HF_TOKEN"] = secret
    del secret
except Exception:
    pass  # Public model can be downloaded without a token.

print("Working directory:", ROOT)
print("Training: Subjectesis", "+ optional answer-only baseline" if TRAIN_BASELINE else "only")
print("Model: official Qwen3.5-4B. Precision: explicit NF4 QLoRA.")

Verified checkpoints restored from the recovery archive.
Working directory: /kaggle/working/subjectesis_qwen35_4b_t4x2_v1
Training: Subjectesis only
Model: official Qwen3.5-4B. Precision: explicit NF4 QLoRA.


## 2. Rebuild the exact data and run CPU checks

This performs the original **31 data tests**, verifies all frozen hashes, and runs
CPU tests for masking, gradients, accumulation, recovery, and the 4B changes.
It does not perform GPU/model-quality evaluation or open test answers for training.


In [3]:
try:
    stream([sys.executable, ROOT / "code/prepare.py", "--root", ROOT],
           ROOT / "logs/prepare.log", env=child_env(ROOT))
    for name in ["tests_cpu.py", "test_4b_adaptation.py"]:
        stream([sys.executable, ROOT / "code" / name],
               ROOT / "logs" / (name + ".log"), env=child_env(ROOT))
except BaseException:
    print("Diagnostic ZIP:", package_outputs(ROOT))
    raise
print("Dataset and CPU checks passed.")

Starting: /usr/bin/python3 /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/code/prepare.py --root /kaggle/working/subjectesis_qwen35_4b_t4x2_v1
test_all_hashes (__main__.DataTests.test_all_hashes) ... ok
test_appraisal_adjudications (__main__.DataTests.test_appraisal_adjudications) ... ok
test_assistant_mask_and_padding (__main__.DataTests.test_assistant_mask_and_padding) ... ok
test_audit_comments_never_become_model_targets (__main__.DataTests.test_audit_comments_never_become_model_targets) ... ok
test_balanced_revision_controls (__main__.DataTests.test_balanced_revision_controls) ... ok
test_checksum_tampering (__main__.DataTests.test_checksum_tampering) ... ok
test_citation_presence_does_not_determine_unknown (__main__.DataTests.test_citation_presence_does_not_determine_unknown) ... ok
test_claims_and_exact_citations (__main__.DataTests.test_claims_and_exact_citations) ... ok
test_complete_annotation_coverage (__main__.DataTests.test_complete_annotation_coverage) ... ok
test_desire_co

## 3. Install the pinned packages and test the installed Qwen implementation

Targets the Python 3.12 / PyTorch 2.10.0 environment in your Kaggle logs. Torch and its
CUDA installation are retained. Project packages, including the TorchAO fix, take
priority inside `torch_env`. Unrelated host-package warnings are logged; actual imports,
LoRA insertion and a tiny backward pass must pass before model download.

The tiny CPU model also tests the 4B-style tied checkpoint and remapped text weights.
It uses random tiny weights, not the real pretrained 4B model.


In [4]:
try:
    stream([sys.executable, ROOT / "code/setup_runtime.py", "--root", ROOT],
           ROOT / "logs/setup.log", env=child_env(ROOT))
    ENV_PYTHON = ROOT / "torch_env/bin/python"
    stream([ENV_PYTHON, ROOT / "code/tiny_qwen_check.py", "--root", ROOT],
           ROOT / "logs/tiny_installed_qwen.log", env=child_env(ROOT))
except BaseException:
    print("Diagnostic ZIP:", package_outputs(ROOT))
    raise


Starting: /usr/bin/python3 /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/code/setup_runtime.py --root /kaggle/working/subjectesis_qwen35_4b_t4x2_v1
Attached GPUs: [
  {
    "gpu": 0,
    "name": "Tesla T4",
    "capability": [
      7,
      5
    ],
    "free_gib": 14.45977783203125,
    "total_gib": 14.56219482421875
  },
  {
    "gpu": 1,
    "name": "Tesla T4",
    "capability": [
      7,
      5
    ],
    "free_gib": 14.45977783203125,
    "total_gib": 14.56219482421875
  }
]
Installing project packages; retaining host Torch and CUDA.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 268.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 381.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 351.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 251.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 312.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/

## 4. Use the Qwen tokenizer and download the 4B checkpoint

Every prompt/completion boundary is checked with the actual selected tokenizer.
Over-length examples cause a clear stop, never truncation. Both workers share one
on-disk model download. Only selected model/tokenizer files are downloaded.

The disk check keeps a 4 GiB reserve and uses the 4B file sizes, not the old 9B threshold.
If space is short, do not bypass the check or delete training outputs blindly.


In [5]:
try:
    stream([ENV_PYTHON, ROOT / "code/tokenize_data.py", "--root", ROOT],
           ROOT / "logs/tokenize.log", env=child_env(ROOT))
    stream([ENV_PYTHON, ROOT / "code/download_model.py", "--root", ROOT],
           ROOT / "logs/model_download.log", env=child_env(ROOT))
except BaseException:
    print("Diagnostic ZIP:", package_outputs(ROOT))
    raise


Starting: /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/torch_env/bin/python /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/code/tokenize_data.py --root /kaggle/working/subjectesis_qwen35_4b_t4x2_v1
Tokenized subjectesis: 5000/30266
Process alive: 30s. Full log: tokenize.log
Tokenized subjectesis: 10000/30266
Process alive: 60s. Full log: tokenize.log
Tokenized subjectesis: 15000/30266
Tokenized subjectesis: 20000/30266
Process alive: 90s. Full log: tokenize.log
Tokenized subjectesis: 25000/30266
Tokenized subjectesis: 30000/30266
subjectesis {'fingerprint': 'ff7030c21cee5dd86d30fc5520b64792918907736e9349c2edd00c1261562266', 'source_sha256': 'e99bcf7049c6e5fb831947480fa27f51ab274cb8b46e2a76fa81030b0a2272ea', 'array_sha256': '14a4e3f06bc307fefe0b0bb9c0b471ea3d6dadd81c539327a6b9fde7ed4be482', 'examples': 30266, 'total_tokens': 20255604, 'supervised_tokens': 3712246, 'max_tokens': 1621, 'longest_index': 21245, 'truncations': 0}
Exact Qwen tokenizer and response masks checked. Test labels n

## 5. Actual 4B GPU check, then full training

Two synchronized workers first run the longest-example backward/update and adapter
save/reload check. Those test updates are discarded. Then training begins automatically.

The numerical recipe remains NF4 QLoRA, rank 8, seed 42, learning rate 1e-4, effective
batch size 8, one epoch by default. All rows are visited, including the short final batch.
The 4B head shares the pretrained embedding matrix; it is not randomly initialized.

Logs distinguish model loading, smoke checks, validation and actual optimizer steps.
Checkpoints are saved every 25 steps. Validation runs every 500 steps and at completion.
The running time estimate is based on measured progress, not a promised completion time.

**Validation is answer-letter prediction without the Subjectesis runtime review loop.**
Final sealed test, external transfer and with/without-review comparisons remain separate.

If `paused_resumable` appears, rerun this cell to resume in the same session.


In [6]:
# Run this new cell immediately BEFORE the existing training cell 5.
from pathlib import Path
import hashlib
import json

ROOT = Path(globals().get(
    "ROOT", "/kaggle/working/subjectesis_qwen35_4b_t4x2_v1"
))
path = ROOT / "code/engine.py"

if not path.is_file():
    raise RuntimeError("The existing 4B notebook source was not found.")

settings = json.loads((ROOT / "settings.json").read_text())
if settings.get("model_id") != "Qwen/Qwen3.5-4B":
    raise RuntimeError("This patch is for the Qwen3.5-4B notebook only.")

original = path.read_text()
marker = "# SUBJECTESIS_TRAIN_MODE_FIX_V1"
anchor = (
    "    import torch.distributed as dist\n"
    "    use_dist=dist.is_available() and dist.is_initialized()"
)

replacement = '''    # SUBJECTESIS_TRAIN_MODE_FIX_V1
    # Loading a pretrained model leaves decoder layers in eval mode.
    # The HF checkpoint path needs BOTH gradient_checkpointing and training.
    # Calling train() on a NEW wrapper's parent is required to propagate to children.
    ddp.train()
    wrapped = ddp.module if hasattr(ddp, "module") else ddp
    if isinstance(wrapped, CompletionLoss):
        layers = getattr(wrapped.core().model, "layers", ())
        if layers and any(hasattr(layer, "gradient_checkpointing") for layer in layers):
            active = sum(
                bool(getattr(layer, "gradient_checkpointing", False)) and layer.training
                for layer in layers
            )
            if active != len(layers):
                raise RuntimeError(
                    f"Decoder activation checkpointing is active on {active}/{len(layers)} layers. "
                    "Stopping before a memory-heavy forward pass."
                )
            if not getattr(wrapped, "_training_mode_fix_logged", False):
                print(
                    f"TRAIN MODE ON: activation checkpointing enabled on {active}/{len(layers)} decoder layers.",
                    flush=True,
                )
                wrapped._training_mode_fix_logged = True
    import torch.distributed as dist
    use_dist=dist.is_available() and dist.is_initialized()'''

if marker not in original:
    if original.count(anchor) != 1:
        raise RuntimeError("Unexpected engine.py version; no files changed.")

    # Do not silently change code underneath an existing trained checkpoint.
   # saved_runs = list((ROOT / "runs").glob("*/latest.json"))
    #saved_runs += list((ROOT / "runs").glob("*/completed.json"))
    #if saved_runs:
     #   raise RuntimeError(
      #      "A saved training run exists. Preserve it before changing its code."
       # )

    patched = original.replace(anchor, replacement, 1)
    compile(patched, str(path), "exec")

    backup = path.with_name("engine.py.before_training_mode_fix")
    if not backup.exists():
        backup.write_text(original)

    temporary = path.with_name("engine.py.tmp")
    temporary.write_text(patched)
    temporary.replace(path)

    report = ROOT / "reports/training_mode_fix.json"
    report.parent.mkdir(parents=True, exist_ok=True)
    report.write_text(json.dumps({
        "fix": "recursive train mode before gradient-bearing updates",
        "before_sha256": hashlib.sha256(original.encode()).hexdigest(),
        "after_sha256": hashlib.sha256(patched.encode()).hexdigest(),
        "dataset_and_settings_unchanged": True,
    }, indent=2))
else:
    print("Training-mode patch is already installed.")

print("PATCH APPLIED. Rerun the existing cell 5. Do not rerun cells 1-4.")

PATCH APPLIED. Rerun the existing cell 5. Do not rerun cells 1-4.


In [7]:
command = [ENV_PYTHON, "-u", ROOT / "code/launch.py", "--root", ROOT]
if SMOKE_ONLY:
    command.append("--smoke-only")
stream(command, ROOT / "logs/session.log", env=child_env(ROOT))


Starting: /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/torch_env/bin/python -u /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/code/launch.py --root /kaggle/working/subjectesis_qwen35_4b_t4x2_v1
Starting: /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/torch_env/bin/python -u -m torch.distributed.run --standalone --nnodes=1 --nproc_per_node=2 /kaggle/working/subjectesis_qwen35_4b_t4x2_v1/code/train.py --root /kaggle/working/subjectesis_qwen35_4b_t4x2_v1 --condition subjectesis
[W919 10:13:56.254821550 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W919 10:13:59.417611226 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W919 10:13:59.475820478 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-li

KeyboardInterrupt: 

hmyyyyhbh

df

## 6. Save the output and resume

Download **subjectesis_qwen35_4b_outputs.zip** before ending the session. It contains
adapter checkpoints, optimizer/RNG progress and diagnostics, not the base weights.

For another Kaggle session, attach this **4B edition's** ZIP privately, set `RESUME_ARCHIVE`
in section 1 to its `/kaggle/input/...` path, and Run All with the same settings.
Old 9B/Gemma checkpoints cannot resume this model. The most recent two checkpoints are retained.

The free runtime can be interrupted; a hard shutdown can lose progress after the latest save.


In [8]:
for condition in (["subjectesis", "ordinary"] if TRAIN_BASELINE else ["subjectesis"]):
    status = ROOT / "runs" / condition / "status.json"
    print(condition, read_json(status) if status.exists() else "No training status yet")
archive = package_outputs(ROOT)
print("Output ZIP:", archive)
from IPython.display import FileLink, display
display(FileLink(str(archive)))


subjectesis No training status yet
Output ZIP: /kaggle/working/subjectesis_qwen35_4b_outputs.zip


/kaggle/working/subjectesis_qwen35_4b_outputs.zip

In [9]:
from pathlib import Path
import sys, shutil, zipfile, time
from IPython.display import FileLink, display

ROOT = Path("/kaggle/working/subjectesis_qwen35_4b_t4x2_v1")

sys.path.insert(0, str(ROOT / "code"))
from common import package_outputs, read_json

# Rebuild the ZIP from the CURRENT checkpoint state
archive = Path(package_outputs(ROOT))

latest_file = ROOT / "runs" / "subjectesis" / "latest.json"
latest = read_json(latest_file) if latest_file.exists() else {}

step = latest.get("step", "unknown")
examples = latest.get("examples_seen", "unknown")

# Give it a fresh filename so Kaggle/browser caching cannot confuse it
fresh = Path(
    f"/kaggle/working/subjectesis_checkpoint_step_{step}.zip"
)
shutil.copy2(archive, fresh)

# Verify ZIP is healthy before you download it
with zipfile.ZipFile(fresh) as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"Corrupted ZIP member: {bad}")

print("Checkpoint step:", step)
print("Examples seen:", examples)
print("File:", fresh)
print("Size MB:", round(fresh.stat().st_size / 1024**2, 1))

display(FileLink(str(fresh)))

Checkpoint step: unknown
Examples seen: unknown
File: /kaggle/working/subjectesis_checkpoint_step_unknown.zip
Size MB: 401.0


/kaggle/working/subjectesis_checkpoint_step_unknown.zip

## Verification scope and official references

The authoring environment reran CPU/data checks and a real two-process CPU distributed
update test. These do not establish pretrained 4B performance, CUDA/bitsandbytes compatibility,
GPU fit, or completion time. The notebook tests those available hardware operations on Kaggle.
The maintained PyTorch reference DeltaNet path is used on T4; it can be slower than newer-GPU kernels.

- [Official post-trained Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B)
- [Pinned model revision](https://huggingface.co/Qwen/Qwen3.5-4B/commit/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a)
- [Qwen3.5 text model and hybrid attention](https://huggingface.co/docs/transformers/v5.13.0/model_doc/qwen3_5)
- [Bitsandbytes quantization](https://huggingface.co/docs/transformers/v5.13.0/quantization/bitsandbytes)
- [PEFT quantization training](https://huggingface.co/docs/peft/developer_guides/quantization)

Data annotations remain assistant-authored, not independently human-validated. Exact source
messages, evidence/revision tasks, eligibility, and held-out files are preserved. A trained
adapter is not by itself proof that the Subjectesis hypothesis is supported.
